# Pion Stop Inference (Unified Config Pipeline)

Run pion-stop inference from aligned upstream prediction shards using the unified config-driven inference pipeline.

## 1) Environment Setup

In [1]:
from pathlib import Path

from pioneerml.integration.zenml import load_step_output
from pioneerml.integration.zenml import utils as zenml_utils
from pioneerml.pipeline.pipelines.inference import inference_pipeline

PROJECT_ROOT = zenml_utils.find_project_root()
zenml_utils.setup_zenml_for_notebook(root_path=PROJECT_ROOT, use_in_memory=True)

Using ZenML repository root: /workspace
Ensure this is the top-level of your repo (.zen must live here).


## 2) Select and Align Input Files

In [2]:
def _pick_pred(pred_dir: Path, main_path: Path) -> Path | None:
    candidates = [
        pred_dir / f"{main_path.stem}_preds.parquet",
        pred_dir / f"{main_path.stem}_preds_latest.parquet",
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def resolve_aligned_inputs(*, project_root: str, max_files: int | None = 1) -> dict:
    data_dir = Path(project_root) / "data"
    main_paths = sorted(data_dir.glob("ml_output_*.parquet"))
    if max_files is not None:
        main_paths = main_paths[: int(max_files)]
    if not main_paths:
        raise RuntimeError(f"No main parquet files found in {data_dir}.")

    dirs = {
        "group_probs": data_dir / "group_classifier",
        "group_splitter": data_dir / "group_splitter",
        "endpoint": data_dir / "endpoint_regressor",
        "event_splitter": data_dir / "event_splitter",
    }

    aligned: list[tuple[Path, Path, Path, Path, Path]] = []
    for main in main_paths:
        gp = _pick_pred(dirs["group_probs"], main)
        gs = _pick_pred(dirs["group_splitter"], main)
        ep = _pick_pred(dirs["endpoint"], main)
        es = _pick_pred(dirs["event_splitter"], main)
        if not (gp and gs and ep and es):
            missing = []
            if gp is None:
                missing.append("group_classifier")
            if gs is None:
                missing.append("group_splitter")
            if ep is None:
                missing.append("endpoint_regressor")
            if es is None:
                missing.append("event_splitter")
            raise RuntimeError(f"Missing aligned predictions for {main.name}: {', '.join(missing)}")
        aligned.append((main, gp, gs, ep, es))

    return {
        "main_sources": [str(m.resolve()) for (m, _, _, _, _) in aligned],
        "optional_sources_by_name": {
            "group_probs": [str(gp.resolve()) for (_, gp, _, _, _) in aligned],
            "group_splitter": [str(gs.resolve()) for (_, _, gs, _, _) in aligned],
            "endpoint": [str(ep.resolve()) for (_, _, _, ep, _) in aligned],
            "event_splitter": [str(es.resolve()) for (_, _, _, _, es) in aligned],
        },
        "source_type": "file",
    }


input_sources_spec = resolve_aligned_inputs(project_root=PROJECT_ROOT, max_files=1)
model_path = None
output_dir = str((Path(PROJECT_ROOT) / "data" / "pion_stop_regression").resolve())

print(f"Inference shards: {len(input_sources_spec['main_sources'])}")
for src in input_sources_spec["main_sources"]:
    print(" -", src)
print("output_dir:", output_dir)

Inference shards: 1
 - /workspace/data/ml_output_000.parquet
output_dir: /workspace/data/pion_stop_regression


## 3) Reusable Config Helpers

In [3]:
from pioneerml.plugin.runtime import ensure_plugins_loaded
ensure_plugins_loaded()

from pioneerml_base_plugin.pion_stop.pipeline import load_config
from pioneerml_base_plugin.utils.config_loader import with_loader_sources, with_model_handle_path, with_writer_output


## 4) Build Step Config Blocks

In [4]:
pipeline_config = load_config()["inference"]
pipeline_config = with_loader_sources(
    pipeline_config,
    main_sources=input_sources_spec["main_sources"],
    optional_sources_by_name=input_sources_spec["optional_sources_by_name"],
)
pipeline_config = with_model_handle_path(pipeline_config, model_path=model_path)
pipeline_config = with_writer_output(
    pipeline_config,
    output_dir=output_dir,
)


## 5) Assemble `pipeline_config` and Run

In [5]:
run = inference_pipeline.with_options(enable_cache=False)(
    pipeline_config=pipeline_config,
)


Initiating a new run for the pipeline: inference_pipeline.
Caching is disabled by default for inference_pipeline.
Using user: default
Using stack: default
  deployer: default
  artifact_store: default
  orchestrator: default
You can visualize your pipeline runs in the ZenML Dashboard. In order to try it locally, please run zenml login --local.
Step build_model_handle has started.
[build_model_handle] No materializer is registered for type <class 'pioneerml.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'>, so the default Pickle materializer was used. Pickle is not production ready and should only be used for prototyping as the artifacts cannot be loaded when running with a different Python version. Please consider implementing a custom materializer for type <class 'pioneerml.integration.pytorch.model_handles.torchscript_model_handle.TorchScriptModelHandle'> according to the instructions at https://docs.zenml.io/concepts/artifacts/materializers
Step bui

## 6) Inspect Inference Outputs

In [6]:
inference_output = load_step_output(run, "run_inference")
print("inference_output:", inference_output)

predictions_paths = [Path(p) for p in (inference_output.get("predictions_paths") or [])]
if not predictions_paths and inference_output.get("predictions_path"):
    predictions_paths = [Path(inference_output["predictions_path"])]

print("predictions_paths:")
for p in predictions_paths:
    print(" ", p)

inference_output: {'predictions_path': '/workspace/data/pion_stop_regression/ml_output_000_preds.parquet', 'predictions_paths': ['/workspace/data/pion_stop_regression/ml_output_000_preds.parquet'], 'timestamped_predictions_path': None, 'timestamped_predictions_paths': []}
predictions_paths:
  /workspace/data/pion_stop_regression/ml_output_000_preds.parquet


## 7) Optional: Quick Parquet Validation

In [7]:
import gc

import pyarrow as pa
import pyarrow.parquet as pq

if not predictions_paths:
    raise RuntimeError("No prediction parquet files were exported.")

pf = pq.ParquetFile(predictions_paths[0])
print("file:", predictions_paths[0])
print("rows:", pf.metadata.num_rows)
print(pf.schema_arrow)

if pf.num_row_groups > 0:
    sample = pf.read_row_group(0).slice(0, 3)
    print(sample)
else:
    sample = None
    print("No row groups found.")

del sample, pf
gc.collect()
pa.default_memory_pool().release_unused()

file: /workspace/data/pion_stop_regression/ml_output_000_preds.parquet
rows: 1024
event_id: int64
pred_pion_stop_x: list<element: float>
  child 0, element: float
pred_pion_stop_x_q16: list<element: float>
  child 0, element: float
pred_pion_stop_x_q50: list<element: float>
  child 0, element: float
pred_pion_stop_x_q84: list<element: float>
  child 0, element: float
pred_pion_stop_y: list<element: float>
  child 0, element: float
pred_pion_stop_y_q16: list<element: float>
  child 0, element: float
pred_pion_stop_y_q50: list<element: float>
  child 0, element: float
pred_pion_stop_y_q84: list<element: float>
  child 0, element: float
pred_pion_stop_z: list<element: float>
  child 0, element: float
pred_pion_stop_z_q16: list<element: float>
  child 0, element: float
pred_pion_stop_z_q50: list<element: float>
  child 0, element: float
pred_pion_stop_z_q84: list<element: float>
  child 0, element: float
time_group_ids: list<element: int64>
  child 0, element: int64
pyarrow.Table
event_id: